# Build Different Cherry Tables 

## Imports

In [20]:
import pickle
import random
from hashlib import sha256
import mmh3
from tqdm import tqdm
from math import pi, sqrt, e, log
import time
import csv

## Table Parameters

### Initialise Startpoints

In [2]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

### Parameters

In [21]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)
##############################################################################
# cost factor used for Kis - 5 to 30 in steps of 5
cost_factors = list(range(5, 31, 5))
cost = 5

### Cherry-picks per column - Kis

In [ ]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and the different cost factors
all_Kis = []
for i in range(len(cost_factors)):
    all_Kis.append(data[-1][i][0])

# record simulated m_t and costs
# each in a tuple (simulated_mt, cost)
simulated = []
for i in range(len(cost_factors)):
    # m_t is last m_values value
    simulated.append((round(data[-1][i][3][-1]), data[-1][i][2]))

Kis = data[-1][0][0]

### Points to Search

In [ ]:
# load points to search from pickle file
with open(f'to_search_N_{nlabel}.pkl', 'rb') as f:
    to_search = pickle.load(f)
    f.close()

## Hash and Reduction Functions

In [5]:
def H(x):
    return sha256(x.to_bytes(8, 'little')).digest()  # return bytes directly

def r(y, i, ell=0):
    return mmh3.hash(y, i + ell*t, signed=False) % N

## Place to Store Results

In [ ]:


# create the file and add header - for searching the table
# with open("building_table_results.csv", "w", newline="") as fsearching:
#     searching_writer = csv.writer(fbuilding)
#     searching_writer.writerow(["cost_factor", "simulated_mt", "actual_mt", "simulated_cost", "hashes", "reductions", "duration"])  # headers

## Building the Table

In [6]:
# build cherry table - store how many hashes and reductions are done
# returns: cherry table, indexes, no. hashes, no. reductions, time to build
def build_cherry_table(t, alpha, startpoints, Kis, cost):
    # parameters to store how many hashes and reductions are done, as well as actual time it takes to make this table
    hashes = 0
    reductions = 0
    duration = 0

    # start monitoring time
    start = time.perf_counter()


    # store table in dictionary
    # initialise the table with sp:sp pairs
    table = {sp: sp for sp in startpoints}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # for each column
    for i in tqdm(range(t), desc=f"Calculating columns: "):

        # variable to store best cherry-pick
        best_trial = -1

        # hash all current points then store with startpoints - this stores all our current points
        hashed_points = {H(mi): sp for mi, sp in table.items()}
        hashes += (len(startpoints))    # increment hashes count

        # we are going to continuously replace table with the best rf trial, so we empty it for now
            # we haven't lost the current points as we have them hashed in the hashed_points dictionary
        table = {}

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # trial all the reduction functions for this column
        # for rf_trial in tqdm(range(k_i), desc=f"Choosing best RF: "):
        for rf_trial in range(k_i):

            # create a trial column to store results of current trial
            trial_column = {}

            # go through each key in hashed_points and store its reduction with sp
            for x in hashed_points:
                # reduce the hash and store in column
                trial_column[r(x, rf_trial)] = hashed_points[x]
                reductions += 1

            # if trial_column is bigger than current table stored, we replace it
            if len(trial_column) > len(table):
                # replace it 
                table = trial_column
                # replace best cherry-pick
                best_trial = rf_trial

        # now store the best rf cherry pick
        rf_indexes.append(best_trial)


    # finished making table so stop recording time
    duration = time.perf_counter() - start

    # store the table as a pickle file
    with open(f'constructed_tables/cherry_table_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'wb') as f:
        pickle.dump(table, f)
        f.close()

    # store the indexes as a pickle file
    with open(f'constructed_tables/cherry_indexes_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'wb') as f:
        pickle.dump(rf_indexes, f)
        f.close()


    # finished, so return table and rf indexes
    return table, rf_indexes, hashes, reductions, duration

## Searching the Table

### Function to help hash and reduce accordingly to continue search

In [15]:
# function to continue search
# we need to take in what column we're at (c), our key (y), # columns (t), current total of hashes and reductions
def continue_cherry_search(y, t, c, hashes, reductions, indexes):
    # reduce c by 1 to move to the previous column
    c -= 1
    # number of columns between current column and end 
    # diff = t - c  
    # reduce y by the new c index
    x = r(y, indexes[c])
    reductions += 1
    # then hash and reduce however many times to move back through columns
    for i in range((t - c) - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
        x = r(H(x), indexes[t-i])
        hashes += 1
        reductions += 1

    # return x, c, hashes, reductions
    return x, c, hashes, reductions

### Searching vanilla table function

In [57]:
def search_cherry_table(y, t, table, indexes):
    # keep track of hashes, reductions, no. false alarms, cols false alarms occur in, duration
    cols_searched = 0
    hashes = 0
    reductions = 0
    false_alarms = 0
    false_alarm_cols = []
    duration = 0

    # start monitoring time
    start = time.perf_counter()

    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table 
    x = r(y, indexes[c])
    reductions += 1

    # while we haven't reached the end of our chain
    while c > -1:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        if (x in table.keys()):
            point = table[x]   # search for endpoint in table and get startpoint
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), indexes[i])
                hashes += 1
                reductions += 1
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                hashes += 1
                duration = time.perf_counter() - start
                return True, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration
            
            # if that didn't work then we ran into a false alarm
            else:
                false_alarms += 1
                false_alarm_cols.append(c)
                # continue the search
                x, c, hashes, reductions = continue_cherry_search(y, t, c, hashes, reductions, indexes)
 
        # if we didn't find a match in endpoints, we need to restart the search
        else:
            # hash and reduce the ciphertext accordingly
            x, c, hashes, reductions = continue_cherry_search(y, t, c, hashes, reductions, indexes)

        cols_searched += 1

    # finished search so stop recording time
    duration = time.perf_counter() - start

    # We have searched all columns - return -1
    return False, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration

## Run

### Get the table

In [41]:
# either build or load table
def get_cherry_table(alpha, t, cost):
    # try loading table from pickle file
    try:
        with open(f'constructed_tables/cherry_table_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'rb') as f:
            table = pickle.load(f)
            f.close()

        with open(f'constructed_tables/cherry_indexes_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'rb') as f:
            indexes = pickle.load(f)
            f.close()

    # if no pickle file found, build the table
    except FileNotFoundError:
        Kis = data[-1][cost_factors.index(cost)][0]
        table, indexes = build_cherry_table(t, alpha, startpoints, Kis, cost)

    return table, indexes

### Get data for building different tables

In [ ]:
# set up loop to build different tables with different cost factors
# create the file and add header - for building the table
with open("building_table_results.csv", "w", newline="") as fbuilding:
    building_writer = csv.writer(fbuilding)
    building_writer.writerow(["cost_factor", "simulated_mt", "actual_mt", "simulated_cost", "hashes", "reductions", "duration"])  # headers
    
    for i in range(len(cost_factors)):
        cost = cost_factors[i]
        Kis = all_Kis[i]
        simulated_mt, simulated_cost  = simulated[i]

        # build cherry table
        table, indexes, hashes, reductions, duration = build_cherry_table(t, alpha, startpoints, Kis, cost)

        # store results in csv file
        # ["cost_factor", "simulated_mt", "actual_mt", "simulated_cost", "hashes", "reductions", "duration"]
        building_writer.writerow([cost, simulated_mt, len(table), simulated_cost, hashes, reductions, duration])

Calculating columns: 100%|██████████| 80/80 [05:21<00:00,  4.02s/it]


### Get data for searching tables

In [ ]:
# store how long each bulk search takes
times = []

# loop to search each individual table
for i in range(len(cost_factors)):
    # load the table and indexes
    table, indexes = get_cherry_table(alpha, t, cost_factors[i])

    # load the data to search - to_search

    # make a csv file to store results
    with open(f"searching_table_results_cost_{cost_factors[i]}.csv", "w", newline="") as fsearching:
        searching_writer = csv.writer(fsearching)
        searching_writer.writerow(["point", "found", "hashes", "reductions", "false_alarms", "false_alarm_cols", "cols_searched", "duration"])  # headers
        
        # start bulk search time monitoring
        bulk_start = time.perf_counter()

        # search each point in to_search
        for point in to_search:
            y = H(point)
            found, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration = search_cherry_table(y, t, table, indexes)
            searching_writer.writerow([point, found, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration])

        # stop bulk search time monitoring
        bulk_duration = time.perf_counter() - bulk_start
        times.append((cost_factors[i], bulk_duration))

# store bulk search times in a pickle file
with open(f'bulk_search_times_N_{nlabel}.pkl', 'wb') as f:
    pickle.dump(times, f)
    f.close()
        

### random tests